# Modelos por cluster: FE con el cluster como nivel de agregación, un modelo por grupo

Toma las etiquetas de `dtw_nuevo` y las usa para **dos cosas distintas**, que conviene no
confundir porque se pueden evaluar por separado:

1. **El cluster como nivel de agregación en el feature engineering.** Igual que hoy
   existen `tn_total_cat1` y los shares por categoría, acá aparece el total del cluster,
   sus lags, y la participación de cada par dentro de su cluster. Esto es una feature
   más: la usa un modelo global, sin partir nada.
2. **Un modelo por cluster.** Cada grupo entrena su propio LightGBM, y al final se
   pegan todas las predicciones.

## Por qué puede funcionar, y por qué puede no

El argumento a favor: los clusters de `dtw_nuevo` agrupan pares con **forma temporal
parecida cruzando categorías** — la `cat3` dominante de cada cluster pesa 0,11–0,16 con
lift ≈ 1, así que no son categorías disfrazadas. Si un cluster junta series estacionales
y otro series planas, un modelo especializado puede aprender la estacionalidad sin que
las planas le diluyan la señal.

El argumento en contra, que es el que gana casi siempre: **partir los datos le quita
filas a cada modelo.** Seis modelos con 1/6 de los datos cada uno pueden ser peores que
uno con todo, porque los árboles ya son capaces de partir por cluster si les das la
etiqueta como feature. Ésa es exactamente la comparación que hace este notebook:

| Rama | Qué es |
|---|---|
| `0_baseline` | media móvil de 3 meses. El piso. |
| `1_global` | un modelo, todas las filas, **sin** la etiqueta de cluster |
| `2_global_con_cluster` | un modelo, todas las filas, **con** `cluster` como feature categórica |
| `3_por_cluster` | un modelo por cluster, predicciones concatenadas |

La comparación que importa es **2 vs 3**: mismas features, misma información
disponible, y la única diferencia es si el árbol decide solo cuándo separar por cluster
o se lo imponemos partiendo los datos. Si `3` no le gana a `2`, partir no sirvió — y eso
es un resultado, no un fracaso.

## El nivel y la entrega

Los clusters son de **pares producto-cliente**, así que el panel y los modelos también.
Kaggle mide por `product_id`, así que la predicción de cada par se **suma sobre los
clientes** al final. Ese paso importa: el error se mide sobre el total del producto, y
al sumar muchos pares los errores de distinto signo se cancelan.

**Los pares sin cluster** (los que `dtw_nuevo` descartó por tener menos de `min_meses`
de historia) van a un grupo propio, `cluster = -1`, y **también entrenan su modelo**. No
se pueden tirar: si un par no tiene predicción, su producto queda incompleto en el
submit. Y no se pueden rellenar con el cluster más común, porque "historia corta" es una
condición real y compartida — es un grupo legítimo, no un faltante.

## 0 — Ambiente

In [ ]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    local = Path(r"C:\Users\Natalia\labo3-bucket")
    if local.is_dir():
        return local
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
DIR_PRE  = BUCKET / "datasets" / "preprocesado"
DIR_FE   = BUCKET / "datasets_fe"
RUTA_EXP = BUCKET / "exp_modelos_cluster"
RUTA_EXP.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"salida : {RUTA_EXP}")
print("\nParquets de cluster disponibles en datasets_fe/:")
for p in sorted(DIR_FE.glob("clusters_pc_*.parquet")):
    print(f"  - {p.name}")

## 1 — Palancas

In [ ]:
def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


PARAM = {
    # ── LAS ETIQUETAS DE CLUSTER ─────────────────────────────────────────
    # Nombre exacto del parquet que dejo dtw_nuevo. None = el mas reciente.
    'archivo_clusters': None,

    # ── FUENTE DE LAS SERIES ─────────────────────────────────────────────
    'fuente': 'preprocesado',            # 'preprocesado' | 'crudo'
    'archivo_preprocesado': None,        # None = el mas grande de datasets/preprocesado/

    # Solo los productos de la lista a entregar. Baja mucho el panel y no cambia el
    # submit: los productos que no hay que entregar no aportan nada al WAPE de Kaggle.
    'solo_productos_target': True,

    # ── PARTICION ────────────────────────────────────────────────────────
    'horizonte': 2,
    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # ── FEATURE ENGINEERING ──────────────────────────────────────────────
    'max_lags': 12,                      # lags del par
    'lags_agregado': 6,                  # lags de los totales (cluster, producto, cliente)
    'ventanas_ma': (3, 6, 12),

    # Niveles sobre los que se calculan totales y shares. 'cluster' es el que agrega
    # este notebook; los otros son los de siempre y estan para poder medir si el
    # cluster aporta algo sobre ellos.
    'niveles_share': ('cluster', 'producto', 'cliente', 'cat3'),

    # ── MUESTREO DE CLIENTES (igual que 03/04) ───────────────────────────
    # Los ids estan ordenados por importancia: los N mas bajos son los N mas grandes.
    # None = todos. OJO: la INFERENCIA nunca se filtra, el submit no puede tener huecos.
    'top_clientes': None,

    # ── MODELOS POR CLUSTER ──────────────────────────────────────────────
    # Un cluster con pocas filas no puede sostener su propio modelo: sobreajusta y
    # ademas ni siquiera va a ver todos los meses. Por debajo de este umbral, el
    # cluster se predice con el modelo global.
    'min_filas_cluster': 20_000,

    # ── QUE RAMAS CORRER ─────────────────────────────────────────────────
    # Las dos ramas globales entrenan con TODAS las filas del panel (~9M con todos los
    # clientes), y son las que se comen la memoria y el tiempo. Las de cluster entrenan
    # con una fraccion cada una, asi que corren aunque las globales no entren.
    #
    #   '0_baseline'            tn_ma3. Gratis, siempre se calcula.
    #   '1_global'              un modelo, todas las filas, SIN la etiqueta.
    #   '2_global_con_cluster'  un modelo, todas las filas, CON la etiqueta.
    #   '3_por_cluster'         un modelo por cluster.
    #
    # LO QUE SE PIERDE al sacar las globales: son las que contestan si el clustering
    # sirve. Sin la 2 no hay con que comparar la 3, y el WAPE de la 3 sola no dice si
    # partir por cluster fue mejor o peor que no hacerlo -- solo dice cuanto da.
    # Para recuperar la comparacion sin pagar los 9M: corre otra vez con
    # top_clientes=20 y las cuatro ramas, y mira ahi el signo de 2 vs 3.
    'ramas': ('0_baseline', '3_por_cluster'),

    # ── OPTUNA: una busqueda POR MODELO ──────────────────────────────────
    # 0 = no buscar, se usan los hiperparametros fijos de 'lgbm'.
    # N > 0 = N trials para CADA modelo: el global sin cluster, el global con cluster,
    # y uno por cada cluster con modelo propio. O sea que el costo total es
    # N x (2 + n_clusters) entrenamientos, mas los finales.
    #
    # Por que por cluster y no una busqueda sola: es el argumento entero de partir por
    # cluster. Si un grupo junta series estacionales y otro series planas, no hay razon
    # para que compartan profundidad ni learning rate -- y si les imponemos los mismos
    # hiperparametros, la rama 3 pelea con una mano atada contra la 2.
    #
    # OJO con el costo: 15 trials x 8 modelos son 120 entrenamientos solo para la
    # busqueda. Arranca con pocos y subilos si el leaderboard lo justifica.
    'optuna_trials': 15,

    # Los trials se acumulan en un sqlite por experimento: si cortas la corrida y la
    # volves a largar, no se pierde lo buscado.
    'optuna_persistente': True,

    # LightGBM. regression_l1 (MAE) porque la metrica es WAPE, que es error absoluto:
    # optimizar MSE castiga distinto de lo que mide la competencia.
    # Con optuna_trials > 0 esto es solo el punto de partida y el respaldo de los
    # clusters que no llegan a tener modelo propio.
    'lgbm': dict(objective='regression_l1', metric='mae', n_estimators=600,
                 learning_rate=0.05, num_leaves=63, min_child_samples=40,
                 subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
                 verbosity=-1, n_jobs=-1, deterministic=True, force_row_wise=True),

    # ── ENTREGA ──────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': True,
    'semilla': 102191,
    'sufijo': '',
}

H = PARAM['horizonte']
L = PARAM['max_lags']
LA = PARAM['lags_agregado']
CATS = ['cat1', 'cat2', 'cat3', 'brand']
KEYS = ['product_id', 'customer_id']

# ── El parquet de clusters ───────────────────────────────────────────────
_disp = sorted(DIR_FE.glob("clusters_pc_*.parquet"))
if not _disp:
    raise FileNotFoundError(
        f"No hay clusters_pc_*.parquet en {DIR_FE}. Corre dtw_nuevo primero.")
if PARAM['archivo_clusters']:
    PATH_CL = DIR_FE / PARAM['archivo_clusters']
    if not PATH_CL.exists():
        raise FileNotFoundError(f"No existe {PATH_CL}.\nDisponibles: "
                                f"{[p.name for p in _disp]}")
else:
    PATH_CL = max(_disp, key=lambda p: p.stat().st_mtime)

EXPERIMENTO = (f"mpc_{PATH_CL.stem.replace('clusters_pc_', '')}"
               f"_{L}lags_min{PARAM['min_filas_cluster']}"
               f"_opt{PARAM['optuna_trials']}"
               f"_r{''.join(sorted(r[0] for r in PARAM['ramas']))}"
               f"_val{PARAM['meses_val'][0]}-{PARAM['meses_val'][-1]}"
               f"_test{PARAM['meses_test'][0]}"
               + (f"_cliTop{PARAM['top_clientes']}" if PARAM['top_clientes'] else "")
               + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"clusters    : {PATH_CL.name}")
print(f"EXPERIMENTO : {EXPERIMENTO}")
print(f"carpeta     : {DIR_OUT.relative_to(BUCKET)}")

## 2 — Carga: series + etiquetas de cluster

Las dos piezas. El panel producto-cliente-mes viene del preprocesado o del crudo, y las
etiquetas del parquet de `dtw_nuevo`. Los pares que no están en ese parquet reciben
`cluster = -1`.

In [ ]:
t0 = time.time()


def a_m(p):
    return (p // 100) * 12 + (p % 100)


def m_a_periodo(m):
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


def normalizar_periodo(df: pl.DataFrame) -> pl.DataFrame:
    """Deja 'periodo' como Int64 AAAAMM: el preprocesado lo graba como Date."""
    dt = df.schema['periodo']
    try:
        temporal = dt.is_temporal()
    except AttributeError:
        temporal = dt in (pl.Date, pl.Datetime)
    if temporal:
        return df.with_columns(
            (pl.col('periodo').dt.year() * 100 + pl.col('periodo').dt.month())
            .cast(pl.Int64).alias('periodo'))
    if dt == pl.Utf8:
        return df.with_columns(
            pl.col('periodo').str.replace_all(r'\D', '').str.slice(0, 6)
              .cast(pl.Int64).alias('periodo'))
    return df.with_columns(pl.col('periodo').cast(pl.Int64))


TARGET_IDS = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt")["product_id"].to_list()

if PARAM['fuente'] == 'preprocesado':
    disp = sorted(DIR_PRE.glob("*.parquet"))
    path_src = (DIR_PRE / PARAM['archivo_preprocesado'] if PARAM['archivo_preprocesado']
                else max(disp, key=lambda p: p.stat().st_size))
    raw = normalizar_periodo(pl.read_parquet(path_src))
    if 'customer_id' not in raw.columns:
        raise ValueError(f"{path_src.name} no tiene customer_id: esta agrupado por "
                         f"producto. Elegi un parquet grpClienteProducto.")
    cats_ok = [c for c in CATS if c in raw.columns]
    prod_cats = raw.select(['product_id'] + cats_ok).unique(subset=['product_id'])
else:
    path_src = DIR_RAW / "sell-in.txt.gz"
    raw = normalizar_periodo(pl.read_csv(path_src, separator="\t"))
    _p = pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t").unique(subset=["product_id"])
    cats_ok = [c for c in CATS if c in _p.columns]
    prod_cats = _p.select(['product_id'] + cats_ok)

print(f"fuente: {path_src.name}")

if PARAM['solo_productos_target']:
    _antes = raw.height
    raw = raw.filter(pl.col('product_id').is_in(TARGET_IDS))
    print(f"solo productos target: {_antes:,} -> {raw.height:,} filas")

if PARAM['top_clientes']:
    _cli = sorted(raw['customer_id'].unique().to_list())[:PARAM['top_clientes']]
    raw = raw.filter(pl.col('customer_id').is_in(_cli))
    print(f"top {PARAM['top_clientes']} clientes: {_cli[0]}..{_cli[-1]} "
          f"-> {raw.height:,} filas")

panel = (raw.group_by(KEYS + ['periodo']).agg(pl.col('tn').sum().alias('tn'))
            .with_columns(a_m(pl.col('periodo')).alias('m')))
del raw
gc.collect()

# ── Las etiquetas ────────────────────────────────────────────────────────
cl = pl.read_parquet(PATH_CL)
COL_CL = next(c for c in cl.columns if c.startswith('cluster_pc_'))
cl = cl.select(KEYS + [pl.col(COL_CL).cast(pl.Int32).alias('cluster')])
print(f"\netiquetas: {cl.height:,} pares en {cl['cluster'].n_unique()} clusters")

M_MIN, M_MAX = int(panel['m'].min()), int(panel['m'].max())
print(f"panel: {panel.height:,} filas   "
      f"meses {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}")
print(f"[{time.time()-t0:.0f}s]")

## 3 — Panel denso y etiqueta de cluster

Se densifica desde la primera venta de cada par hasta el final del panel — la misma
regla que usó `dtw_nuevo` para construir las series que clusterizó, así que las
etiquetas describen exactamente estas series. Los meses sin registro después de la
primera venta son **ceros**, no huecos.

El `join` con las etiquetas es `left`: los pares que `dtw_nuevo` descartó quedan en
`-1`, que es su propio grupo.

In [ ]:
t0 = time.time()

vida = panel.group_by(KEYS).agg(pl.col('m').min().alias('m_nace'),
                                pl.col('m').max().alias('m_ultima'),
                                pl.col('tn').sum().alias('tn_total_par'))

grilla = (vida.select(KEYS + ['m_nace'])
              .with_columns(pl.int_ranges('m_nace', M_MAX + 1).alias('m'))
              .explode('m').select(KEYS + ['m']))

df0 = (grilla.join(panel.select(KEYS + ['m', 'tn']), on=KEYS + ['m'], how='left')
            .with_columns(pl.col('tn').fill_null(0.0))
            .join(vida, on=KEYS, how='left')
            .join(cl, on=KEYS, how='left')
            .with_columns(pl.col('cluster').fill_null(-1),
                          m_a_periodo(pl.col('m')).alias('periodo'),
                          (pl.col('m') - pl.col('m_nace')).alias('edad'))
            .join(prod_cats, on='product_id', how='left')
            .sort(KEYS + ['m']))

_sin = int((df0['cluster'] == -1).sum())
print(f"panel denso: {df0.height:,} filas   "
      f"({df0.select(KEYS).n_unique():,} pares)")
print(f"filas sin cluster (-1): {_sin:,} ({100*_sin/df0.height:.1f}%)")
print("\nfilas por cluster:")
print(df0.group_by('cluster').agg(pl.len().alias('filas'),
                                 pl.col('product_id').n_unique().alias('productos'),
                                 pl.col('customer_id').n_unique().alias('clientes'),
                                 pl.col('tn').sum().round(0).alias('tn'))
        .sort('cluster'))
del panel, grilla
gc.collect()
print(f"[{time.time()-t0:.0f}s]")

## 4 — Feature engineering

Cuatro familias, todas **causales**: cada feature de la fila `t` se calcula con datos
hasta `t` inclusive, nunca con `t+1`.

**Del par** — `tn_lag1..12`, medias móviles 3/6/12, desvío contra la media móvil,
racha de meses sin vender, fracción de meses con venta.

**De los agregados** — para cada nivel de `niveles_share` (cluster, producto, cliente,
cat3) el total de toneladas de ese grupo en el mes `t`, con sus propios lags y media
móvil. Esto es lo que le dice al modelo *cómo se está moviendo el grupo al que pertenece
el par*, que es información que la fila sola no tiene.

**Shares** — la participación del par en cada uno de esos totales, más su lag y su media
móvil. Un share es adimensional y revierte a la media, así que suele ser más predecible
que el nivel.

**Del calendario y del ciclo de vida** — mes del año, edad del par, y si está en sus
primeros meses.

El nivel `cluster` es el que agrega este notebook. Los otros tres están para poder
medir si aporta algo **sobre** ellos: si `sh_cluster` no aparece en la importancia, el
cluster no está diciendo nada que `sh_producto` o `sh_cat3` no dijeran ya.

In [ ]:
V = PARAM['ventanas_ma']

# La clave de agrupacion de cada nivel. 'cluster' es el nivel que agrega este notebook.
NIVELES = {
    'cluster':  ['cluster'],
    'producto': ['product_id'],
    'cliente':  ['customer_id'],
    'cat3':     ['cat3'],
}
NIV = {k: v for k, v in NIVELES.items() if k in PARAM['niveles_share']
       and all(c in df0.columns for c in v)}
SHARES = [f'sh_{n}' for n in NIV]


def div(num, den, alias):
    """Division con guarda: sin esto un grupo con total 0 produce inf/nan."""
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then(pl.col(num) / pl.col(den)).otherwise(0.0).alias(alias))


def agregar_features(d: pl.DataFrame) -> pl.DataFrame:
    """Todas las features, sobre un panel denso ya etiquetado.

    Es una FUNCION y no codigo suelto a proposito: la celda siguiente la llama dos
    veces -- con el panel completo y con el panel cortado en un mes T -- y compara las
    filas <= T. Si alguna feature mirara al futuro, los dos valores no coincidirian.
    Esa es la unica forma de VERIFICAR la causalidad en vez de afirmarla.

    Todo lo que hay aca usa solo datos hasta t inclusive:
      shift(k) con k>0     -> pasado
      rolling_*(w)         -> ventana que TERMINA en t
      cum_max / cum_sum    -> acumulado hasta t
      totales por (grupo, m) -> el mes t, que es contexto conocido en t
    """
    # ── Totales por nivel en el mes t, con sus propios lags y media movil ──
    for nom, claves in NIV.items():
        tot = (d.group_by(claves + ['m']).agg(pl.col('tn').sum().alias(f'tn_{nom}'))
                .sort(claves + ['m']))
        tot = tot.with_columns(
            *[pl.col(f'tn_{nom}').shift(k).over(claves).alias(f'tn_{nom}_lag{k}')
              for k in range(1, LA + 1)],
            pl.col(f'tn_{nom}').rolling_mean(3).over(claves).alias(f'tn_{nom}_ma3'),
        )
        d = d.join(tot, on=claves + ['m'], how='left')

    # ── Shares: participacion del par en cada total ──
    d = d.with_columns(*[div('tn', f'tn_{n}', f'sh_{n}') for n in NIV])

    # ── Historia propia del par ──
    d = d.sort(KEYS + ['m']).with_columns(
        *[pl.col('tn').shift(k).over(KEYS).alias(f'tn_lag{k}') for k in range(1, L + 1)],
        *[pl.col('tn').rolling_mean(w).over(KEYS).alias(f'tn_ma{w}') for w in V],
        *[pl.col(s).shift(1).over(KEYS).alias(f'{s}_lag1') for s in SHARES],
        *[pl.col(s).rolling_mean(3).over(KEYS).alias(f'{s}_ma3') for s in SHARES],
        pl.col('tn').cum_max().over(KEYS).alias('tn_pico'),
        (pl.col('tn') > 0).cast(pl.Int8).alias('vendio'),
    )

    d = d.with_columns(
        *[(pl.col(s) - pl.col(f'{s}_ma3')).alias(f'{s}_dma3') for s in SHARES],
        *[(pl.col('tn') - pl.col(f'tn_ma{w}')).alias(f'tn_dma{w}') for w in V],
        pl.col('vendio').rolling_mean(6).over(KEYS).alias('frac_venta_6'),
        pl.col('vendio').rolling_sum(12).over(KEYS).alias('meses_venta_12'),
        (pl.col('periodo') % 100).alias('mes_del_anio'),
        (pl.col('edad') <= 6).cast(pl.Int8).alias('es_nuevo'),
        pl.when(pl.col('tn_ma3').abs() > 1e-9)
          .then((pl.col('tn') / pl.col('tn_ma3')).clip(0, 10))
          .otherwise(None).alias('idx_vs_ma3'),
        pl.when(pl.col('tn_pico').abs() > 1e-9)
          .then((pl.col('tn') / pl.col('tn_pico')).clip(0, 10))
          .otherwise(None).alias('idx_vs_pico'),
    )

    # racha de meses sin vender: cum_sum de 'vendio' numera los tramos, y dentro de un
    # tramo la primera fila es el mes de la venta -> m - min(m) del tramo es la racha.
    d = d.with_columns(pl.col('vendio').cum_sum().over(KEYS).alias('_g_venta'))
    d = d.with_columns(
        (pl.col('m') - pl.col('m').min().over(KEYS + ['_g_venta'])).alias('racha_sin_venta')
    ).drop('_g_venta')

    # ── El target: toneladas del par H meses despues ──
    d = d.sort(KEYS + ['m']).with_columns(
        pl.col('tn').shift(-H).over(KEYS).alias('clase_tn'),
        m_a_periodo(pl.col('m') + H).alias('periodo_objetivo'),
    )
    return d


t0 = time.time()

# ── Columnas que se BORRAN antes de construir nada ───────────────────────────
# Estas se calcularon sobre la serie COMPLETA del par, o sea que incluyen meses
# posteriores a t. No alcanza con no ponerlas en FEATURES: se van del dataframe, asi
# ninguna feature nueva puede derivarse de ellas por accidente.
#   m_ultima     -> el mes de la ULTIMA venta del par. Saber en 201701 que un par deja
#                   de comprar en 201908 es exactamente el futuro.
#   tn_total_par -> la suma de toda su historia, futuro incluido.
FUGA_SERIE_COMPLETA = ['m_ultima', 'tn_total_par']
df0 = df0.drop([c for c in FUGA_SERIE_COMPLETA if c in df0.columns])
print(f"Borradas por ser agregados de la serie completa: {FUGA_SERIE_COMPLETA}")

df = agregar_features(df0)

_cats = [c for c in CATS if c in df.columns]
df = df.with_columns([pl.col(c).cast(pl.Utf8).fill_null('NA').cast(pl.Categorical)
                      for c in _cats])

# NO_FEAT: identificadores, el eje temporal y el target. 'vendio' sale porque es
# redundante con tn (tn>0) y no aporta; sus derivadas (frac_venta_6, racha) si quedan.
NO_FEAT = {'m', 'm_nace', 'periodo', 'periodo_objetivo', 'clase_tn', 'vendio'}
FEATURES = [c for c in df.columns if c not in NO_FEAT | set(KEYS)]
CAT_FEATURES = _cats + ['cluster']

print(f"\npanel con features: {df.height:,} filas x {len(FEATURES)} features")
print(f"categoricas: {CAT_FEATURES}")
print(f"shares: {SHARES}   niveles: {list(NIV)}")
print(f"[{time.time()-t0:.0f}s]")
gc.collect()

### 4.1 — Auditoría de causalidad (no declarada: verificada)

Dos pruebas mecánicas, y las dos abortan la corrida si fallan.

**Prueba del panel truncado.** Se vuelven a construir TODAS las features sobre un panel
cortado en el último mes de train, y se comparan contra las mismas features del panel
completo, fila por fila, para los meses que existen en los dos. Si una feature usa
cualquier dato posterior a `t`, sus dos versiones difieren y la prueba la nombra. Es la
única forma de verificar causalidad en lugar de afirmarla leyendo el código.

**Prueba de correlación.** Una feature que correlacione más de 0,999 con el target es el
target con otro nombre. Suele pasar por un `shift` con signo equivocado.

In [ ]:
t0 = time.time()

# ── 1. Panel truncado: la prueba fuerte ─────────────────────────────────────
M_CORTE = a_m(max(PARAM['meses_train']))
print(f"Reconstruyendo las features con el panel cortado en "
      f"{max(PARAM['meses_train'])} (m={M_CORTE})...")

# Una muestra de pares: la prueba es exacta, no hace falta el panel entero.
_pares_m = (df0.select(KEYS).unique()
               .sample(n=min(3000, df0.select(KEYS).n_unique()),
                       seed=PARAM['semilla']))
_full = agregar_features(df0.join(_pares_m, on=KEYS, how='inner'))
_trunc = agregar_features(df0.join(_pares_m, on=KEYS, how='inner')
                             .filter(pl.col('m') <= M_CORTE))

_comparables = [c for c in FEATURES
                if c in _trunc.columns and _full.schema[c].is_numeric()]
_a = _full.filter(pl.col('m') <= M_CORTE).sort(KEYS + ['m'])
_b = _trunc.sort(KEYS + ['m'])
if _a.height != _b.height:
    raise RuntimeError(f"la truncada tiene {_b.height} filas y la completa {_a.height} "
                       f"en el mismo rango: el corte no fue limpio")

fugas = []
for c in _comparables:
    x = _a[c].to_numpy().astype(np.float64)
    y = _b[c].to_numpy().astype(np.float64)
    ok = np.isfinite(x) & np.isfinite(y)
    if (np.isnan(x) != np.isnan(y)).any() or (ok.any() and
                                              np.nanmax(np.abs(x[ok] - y[ok])) > 1e-6):
        d = float(np.nanmax(np.abs(x[ok] - y[ok]))) if ok.any() else float('nan')
        fugas.append((c, d))

print(f"  features comparadas : {len(_comparables)}")
print(f"  filas comparadas    : {_a.height:,}")
if fugas:
    print("\n  FUGA DE FUTURO en estas features (difieren al truncar el panel):")
    for c, d in sorted(fugas, key=lambda t: -t[1]):
        print(f"    {c:28s} diferencia maxima {d:.6g}")
    raise RuntimeError(f"{len(fugas)} features usan datos posteriores a t. "
                       f"Sacalas de FEATURES o corregi como se calculan.")
print("  [ok] ninguna feature cambia al ocultar el futuro -> todas son causales")

del _full, _trunc, _a, _b, _pares_m
gc.collect()

# ── 2. Correlacion con el target ────────────────────────────────────────────
_s = df.filter(pl.col('clase_tn').is_not_null()).sample(
    n=min(200_000, df.filter(pl.col('clase_tn').is_not_null()).height),
    seed=PARAM['semilla'])
_y = _s['clase_tn'].to_numpy().astype(np.float64)
sospechosas = []
for c in FEATURES:
    if c in CAT_FEATURES or not df.schema[c].is_numeric():
        continue
    x = _s[c].to_numpy().astype(np.float64)
    ok = np.isfinite(x) & np.isfinite(_y)
    if ok.sum() < 100 or x[ok].std() == 0:
        continue
    r = float(np.corrcoef(x[ok], _y[ok])[0, 1])
    if abs(r) > 0.999:
        sospechosas.append((c, round(r, 6)))
if sospechosas:
    raise RuntimeError(f"Features que son el target disfrazado: {sospechosas}")
print(f"  [ok] ninguna feature correlaciona >0.999 con clase_tn "
      f"(sobre {_s.height:,} filas)")

del _s
gc.collect()
print(f"\nAUDITORIA SUPERADA   [{time.time()-t0:.0f}s]")

## 5 — Control de leakage

Cinco chequeos que abortan la corrida si algo no cierra. El más importante para este
notebook es el tercero: **las etiquetas de cluster no pueden haber visto los meses de
validación**, porque si no la partición en grupos ya sabe algo del futuro y el WAPE
queda inflado.

In [ ]:
sup = df.filter(pl.col('clase_tn').is_not_null())
periodos_sup = sorted(sup['periodo'].unique().to_list())

MESES_TRAIN = [m for m in PARAM['meses_train'] if m in periodos_sup]
MESES_VAL   = [m for m in PARAM['meses_val'] if m in periodos_sup]
MESES_TEST  = [m for m in PARAM['meses_test'] if m in periodos_sup]
MESES_INFER = sorted(df.filter(pl.col('clase_tn').is_null())['periodo'].unique().to_list())[-H:]

errores = []
def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)

print("CONTROL DE LEAKAGE")
print("=" * 76)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, 'train', 'val'),
                     (MESES_VAL, MESES_TEST, 'val', 'test')):
    g = a_m(min(b)) - a_m(max(a))
    chk(g >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {g} >= horizonte {H}")

chk(max(MESES_TRAIN) < min(MESES_VAL) < max(MESES_VAL) < min(MESES_TEST),
    "orden cronologico train < val < test")

# El corte con el que dtw_nuevo genero las etiquetas: tiene que ser <= al primer mes
# de validacion, si no los clusters vieron datos de val.
_cfg_cl = None
for _c in (PATH_CL.parent.parent / 'exp_clusters_pc').rglob('config.json'):
    _t = json.load(open(_c, encoding='utf-8'))
    if f"w{_t.get('window')}" in PATH_CL.stem and _t.get('escalado', '') in PATH_CL.stem:
        _cfg_cl = _t
        break
if _cfg_cl and _cfg_cl.get('mes_corte'):
    chk(a_m(_cfg_cl['mes_corte']) <= a_m(min(MESES_VAL)),
        f"los clusters se calcularon con datos < {_cfg_cl['mes_corte']}, "
        f"y val arranca en {min(MESES_VAL)}")
else:
    print(f"  [aviso] no encontre el config.json de {PATH_CL.name}: verifica a mano "
          f"que dtw_nuevo corrio con mes_corte <= {min(MESES_VAL)}")

chk(not (set(MESES_INFER) & set(MESES_TRAIN + MESES_VAL + MESES_TEST)),
    f"los meses de inferencia {MESES_INFER} no se usan para entrenar ni medir")

# Ninguna feature puede ser el target ni un agregado de la serie completa
chk('clase_tn' not in FEATURES and 'periodo_objetivo' not in FEATURES,
    "el target no esta entre las features")
chk(not [c for c in FUGA_SERIE_COMPLETA if c in FEATURES],
    f"los agregados de serie completa {FUGA_SERIE_COMPLETA} no estan en FEATURES")

# El shift del target es el correcto: clase_tn[i] == tn[i+H] en la serie de un par
_u = sup.group_by(KEYS).agg(pl.len().alias('n')).sort('n', descending=True).head(1)
_s = df.filter((pl.col('product_id') == _u['product_id'][0]) &
               (pl.col('customer_id') == _u['customer_id'][0])).sort('m')
_tn, _cl2 = _s['tn'].to_list(), _s['clase_tn'].to_list()
_mal = [i for i in range(len(_tn) - H)
        if _cl2[i] is not None and abs(_cl2[i] - _tn[i + H]) > 1e-9]
chk(not _mal, f"clase_tn[i] == tn[i+{H}] ({len(_mal)} discrepancias)")

print("=" * 76)
if errores:
    raise RuntimeError(f"Leakage: {errores}")

print(f"TRAIN {len(MESES_TRAIN)} meses ({MESES_TRAIN[0]}..{MESES_TRAIN[-1]})"
      f"  VAL {MESES_VAL}  TEST {MESES_TEST}  INFER {MESES_INFER}")
print(f"filas supervisadas: {sup.height:,}")

## 6 — La métrica

WAPE **en toneladas y agregado por producto**, que es como mide la competencia. El paso
de agregación importa: se suman las predicciones de todos los pares de un producto y se
compara contra la suma real. Un error de +5 en un cliente y −5 en otro se cancela, y
está bien que se cancele, porque a Kaggle sólo le importa el total del producto.

In [ ]:
def wape(y_real, y_pred, ids=None) -> float:
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if ids is not None:
        _, inv = np.unique(np.asarray(ids), return_inverse=True)
        yr = np.bincount(inv, weights=yr)
        yp = np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float('nan') if den == 0 else float(np.abs(yr - yp).sum() / den)


def wape_prod(bloque: pl.DataFrame, pred) -> float:
    """WAPE agregando por product_id, como Kaggle."""
    return wape(bloque['clase_tn'].to_numpy(), pred, bloque['product_id'].to_numpy())


def bloque(meses):
    return sup.filter(pl.col('periodo').is_in(meses))


TR, VA, TE = bloque(MESES_TRAIN), bloque(MESES_VAL), bloque(MESES_TEST)
MESES_FIT_TEST = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
print(f"train {TR.height:,} | val {VA.height:,} | test {TE.height:,} filas")
print(f"productos distintos: train {TR['product_id'].n_unique()} | "
      f"val {VA['product_id'].n_unique()} | test {TE['product_id'].n_unique()}")

## 7 — Las cuatro ramas

`entrenar` y `predecir` son las mismas para todas: lo único que cambia es qué filas ve
cada modelo y qué features se le pasan.

Para la rama por cluster, el criterio de `min_filas_cluster` no es cosmético. Un cluster
con pocas filas entrena un modelo peor que el global **y encima** puede no tener todos
los meses representados, así que ni aprende la estacionalidad. Esos clusters se predicen
con el modelo global, y el notebook imprime cuáles fueron.

In [ ]:
FEAT_SIN_CL = [c for c in FEATURES if c != 'cluster']
FEAT_CON_CL = FEATURES


def entrenar(filas: pl.DataFrame, feats, params=None, semilla=None):
    b = filas.to_pandas()
    p = dict(params if params is not None else PARAM['lgbm'])
    p['seed'] = semilla if semilla is not None else PARAM['semilla']
    cats = [c for c in CAT_FEATURES if c in feats]
    m = lgb.LGBMRegressor(**p)
    m.fit(b[feats], b['clase_tn'], categorical_feature=cats)
    del b
    gc.collect()
    return m


def predecir(modelo, filas: pl.DataFrame, feats):
    return np.maximum(modelo.predict(filas.to_pandas()[feats]), 0.0)


def baseline(filas: pl.DataFrame):
    return np.maximum(filas['tn_ma3'].fill_null(0.0).to_numpy(), 0.0)


# ── El espacio de busqueda ───────────────────────────────────────────────────
def espacio(trial):
    return {
        'objective': 'regression_l1', 'metric': 'mae', 'verbosity': -1,
        'n_jobs': -1, 'deterministic': True, 'force_row_wise': True,
        'subsample_freq': 1,
        'n_estimators':      trial.suggest_int('n_estimators', 150, 1200),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 255),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 300),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }


HIPER = {}          # cache: clave del modelo -> mejores hiperparametros
ESTUDIOS = {}       # clave -> resumen del study, para el resultado.json


def buscar_hiper(clave, filas_tr, filas_va, feats):
    """Optuna minimizando WAPE-producto en VALIDACION, para UN modelo.

    Se busca una sola vez por modelo (con train -> val) y el resultado se cachea. Cuando
    despues se entrena para test, se REUSAN esos hiperparametros y se reentrena con
    train+val: buscar de nuevo usando val ya medida seria elegir hiperparametros contra
    los datos con los que despues se mide.

    Para los modelos por cluster el WAPE se agrega por producto DENTRO de ese cluster.
    Es una suma parcial -- un producto puede tener pares en varios clusters -- asi que
    es un proxy de la metrica final, no la metrica final. Alcanza para ordenar
    hiperparametros, que es lo unico que se le pide a la busqueda.
    """
    if PARAM['optuna_trials'] <= 0:
        return dict(PARAM['lgbm'])
    if clave in HIPER:
        return HIPER[clave]
    if filas_va.height == 0 or filas_tr.height == 0:
        print(f"    [{clave}] sin filas para buscar -> hiperparametros fijos")
        HIPER[clave] = dict(PARAM['lgbm'])
        return HIPER[clave]

    _va_pd = filas_va.to_pandas()

    def obj(trial):
        m = entrenar(filas_tr, feats, espacio(trial))
        p = np.maximum(m.predict(_va_pd[feats]), 0.0)
        del m
        gc.collect()
        v = wape(filas_va['clase_tn'].to_numpy(), p,
                 filas_va['product_id'].to_numpy())
        return float('inf') if not np.isfinite(v) else v

    kw = {}
    if PARAM['optuna_persistente']:
        kw = dict(storage=f"sqlite:///{DB_OPTUNA}", load_if_exists=True)
    st = optuna.create_study(direction='minimize', study_name=f"{EXPERIMENTO}__{clave}",
                             sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
                             **kw)
    t0 = time.time()
    st.optimize(obj, n_trials=PARAM['optuna_trials'])
    HIPER[clave] = {**espacio(optuna.trial.FixedTrial(st.best_params))}
    ESTUDIOS[clave] = {'n_trials': len(st.trials), 'mejor_wape_val': st.best_value,
                       'best_params': st.best_params,
                       'filas_train': int(filas_tr.height),
                       'filas_val': int(filas_va.height)}
    print(f"    [{clave}] {len(st.trials)} trials  mejor WAPE val {st.best_value:.5f}"
          f"  ({time.time()-t0:.0f}s)")
    del _va_pd
    gc.collect()
    return HIPER[clave]


def clusters_con_modelo(meses_fit):
    """(clusters con suficientes filas, clusters que van al global)."""
    tam = bloque(meses_fit).group_by('cluster').agg(pl.len().alias('n')).sort('cluster')
    return (tam.filter(pl.col('n') >= PARAM['min_filas_cluster'])['cluster'].to_list(),
            tam.filter(pl.col('n') < PARAM['min_filas_cluster'])['cluster'].to_list())


def entrenar_por_cluster(meses_fit, feats, buscar=False):
    """Un modelo por cluster, cada uno con SUS hiperparametros."""
    fit = bloque(meses_fit)
    grandes, chicos = clusters_con_modelo(meses_fit)

    modelos = {}
    for c in grandes:
        sub_fit = fit.filter(pl.col('cluster') == c)
        clave = f'cluster_{c}'
        if buscar:
            hp = buscar_hiper(clave, TR.filter(pl.col('cluster') == c),
                              VA.filter(pl.col('cluster') == c), feats)
        else:
            hp = HIPER.get(clave, dict(PARAM['lgbm']))
        t0 = time.time()
        modelos[c] = entrenar(sub_fit, feats, hp)
        print(f"    cluster {c:3d}: {sub_fit.height:8,} filas  [{time.time()-t0:.0f}s]",
              flush=True)
        del sub_fit
        gc.collect()

    m_glob = None
    if chicos:
        hp = (buscar_hiper('global_respaldo', TR, VA, feats) if buscar
              else HIPER.get('global_respaldo', dict(PARAM['lgbm'])))
        m_glob = entrenar(fit, feats, hp)
        print(f"    global (respaldo de los clusters chicos {chicos}): {fit.height:,} filas")
    return modelos, m_glob, chicos


def predecir_por_cluster(modelos, m_glob, filas: pl.DataFrame, feats):
    pred = np.zeros(filas.height, dtype=np.float64)
    idx = np.arange(filas.height)
    cl_col = filas['cluster'].to_numpy()
    for c in np.unique(cl_col):
        mask = cl_col == c
        mod = modelos.get(int(c), m_glob)
        if mod is None:
            raise RuntimeError(f"cluster {c} sin modelo y sin global de respaldo")
        pred[idx[mask]] = predecir(mod, filas.filter(pl.Series(mask)), feats)
    return np.maximum(pred, 0.0)


def correr_ramas(meses_fit, ev: pl.DataFrame, buscar=False):
    out, mods = {}, {}
    quiero = set(PARAM['ramas'])

    # El baseline sale de una columna que ya existe: no cuesta nada y siempre conviene
    # tenerlo, aunque no este pedido. Sin un piso, un WAPE suelto no se puede leer.
    out['0_baseline'] = baseline(ev)

    if '1_global' in quiero:
        print("  1_global (sin cluster)...", flush=True)
        hp1 = buscar_hiper('global_sin_cluster', TR, VA, FEAT_SIN_CL) if buscar \
            else HIPER.get('global_sin_cluster', dict(PARAM['lgbm']))
        m1 = entrenar(bloque(meses_fit), FEAT_SIN_CL, hp1)
        out['1_global'] = predecir(m1, ev, FEAT_SIN_CL)
        mods['1_global'] = m1

    if '2_global_con_cluster' in quiero:
        print("  2_global_con_cluster...", flush=True)
        hp2 = buscar_hiper('global_con_cluster', TR, VA, FEAT_CON_CL) if buscar \
            else HIPER.get('global_con_cluster', dict(PARAM['lgbm']))
        m2 = entrenar(bloque(meses_fit), FEAT_CON_CL, hp2)
        out['2_global_con_cluster'] = predecir(m2, ev, FEAT_CON_CL)
        mods['2_global_con_cluster'] = m2

    if '3_por_cluster' in quiero:
        print("  3_por_cluster:", flush=True)
        mc, mg, chicos = entrenar_por_cluster(meses_fit, FEAT_SIN_CL, buscar=buscar)
        out['3_por_cluster'] = predecir_por_cluster(mc, mg, ev, FEAT_SIN_CL)
        mods['3_por_cluster'] = (mc, mg, chicos)
    return out, mods


DB_OPTUNA = Path.home() / f"optuna_{EXPERIMENTO}.db"
_g_ini, _ch_ini = clusters_con_modelo(MESES_TRAIN)

_modelos_a_buscar = []
if '1_global' in PARAM['ramas']:
    _modelos_a_buscar.append('global_sin_cluster')
if '2_global_con_cluster' in PARAM['ramas']:
    _modelos_a_buscar.append('global_con_cluster')
if '3_por_cluster' in PARAM['ramas']:
    _modelos_a_buscar += [f'cluster_{c}' for c in _g_ini]
    if _ch_ini:
        _modelos_a_buscar.append('global_respaldo')

print(f"RAMAS: {list(PARAM['ramas'])}")
_faltan = {'1_global', '2_global_con_cluster'} - set(PARAM['ramas'])
if _faltan:
    print(f"  OJO: sin {sorted(_faltan)} no se puede medir si el clustering AYUDA, "
          f"solo cuanto da la rama 3.")
print(f"  filas de train por cluster: "
      f"{bloque(MESES_TRAIN).group_by('cluster').agg(pl.len().alias('n')).sort('cluster').to_dicts()}")
if PARAM['optuna_trials'] > 0:
    print(f"\nOPTUNA: {PARAM['optuna_trials']} trials x {len(_modelos_a_buscar)} modelos "
          f"= {PARAM['optuna_trials'] * len(_modelos_a_buscar)} entrenamientos de busqueda")
    print(f"  modelos: {_modelos_a_buscar}")
    print(f"  storage: {DB_OPTUNA if PARAM['optuna_persistente'] else 'en memoria'}")
else:
    print(f"\nOPTUNA desactivado -> hiperparametros fijos para "
          f"{len(_modelos_a_buscar)} modelos")

t0 = time.time()
print("\nVALIDACION (aca se busca)")
pred_val, mod_val = correr_ramas(MESES_TRAIN, VA, buscar=True)
RAMAS = list(pred_val)
wape_val = {r: wape_prod(VA, pred_val[r]) for r in RAMAS}
print(f"[{time.time()-t0:.0f}s]")
for r in RAMAS:
    print(f"  {r:22s} {wape_val[r]:.5f}")

In [ ]:
t0 = time.time()
print("TEST (holdout)")
# buscar=False a proposito: los hiperparametros ya se eligieron con train -> val.
# Volver a buscarlos aca, con test a la vista, seria elegirlos contra el holdout.
pred_test, mod_test = correr_ramas(MESES_FIT_TEST, TE)
print(f"[{time.time()-t0:.0f}s]")

METRICAS = {}
print(f"\n{'rama':24s} {'WAPE val':>10s} {'WAPE test':>10s}")
print("-" * 48)
for r in RAMAS:
    wt = wape_prod(TE, pred_test[r])
    METRICAS[r] = {'val': wape_val[r], 'test': wt}
    print(f"{r:24s} {wape_val[r]:10.5f} {wt:10.5f}")

GANADOR = min(METRICAS, key=lambda r: METRICAS[r]['test'])
_b = METRICAS['0_baseline']['test']
print(f"\nganador en test: {GANADOR}")
for r in RAMAS[1:]:
    print(f"  {r:24s} vs baseline: {100*(_b-METRICAS[r]['test'])/_b:+.1f}%")

# LAS DOS COMPARACIONES QUE JUSTIFICAN EL NOTEBOOK. Solo se pueden calcular si las
# ramas involucradas se corrieron: con 'ramas' recortado, se dice que falta en vez de
# inventar un numero.
_g = METRICAS.get('1_global', {}).get('test')
_gc = METRICAS.get('2_global_con_cluster', {}).get('test')
_pc = METRICAS.get('3_por_cluster', {}).get('test')
APORTE_FEATURE = APORTE_PARTIR = None

if _g is not None and _gc is not None:
    APORTE_FEATURE = 100 * (_g - _gc) / _g
    print(f"\n1) El cluster como FEATURE  (2 vs 1): {APORTE_FEATURE:+.1f}%")
    print("   Positivo = saber a que cluster pertenece el par ayuda al modelo.")
else:
    print("\n1) El cluster como FEATURE: NO MEDIDO "
          "(faltan las ramas 1_global y/o 2_global_con_cluster).")

_ref = _gc if _gc is not None else _g
_nom_ref = '2_global_con_cluster' if _gc is not None else '1_global'
if _ref is not None and _pc is not None:
    APORTE_PARTIR = 100 * (_ref - _pc) / _ref
    print(f"\n2) PARTIR los datos por cluster (3 vs {_nom_ref}): {APORTE_PARTIR:+.1f}%")
    print("   Positivo = los modelos especializados le ganan al que ve todas las filas.")
    print("   Negativo = partir solo le quito filas a cada modelo.")
    if _nom_ref == '1_global':
        print("   OJO: la referencia es 1_global, que NO ve la etiqueta de cluster, asi "
              "que\n   esta comparacion mezcla 'usar la etiqueta' con 'partir los datos'.")
else:
    print("\n2) PARTIR los datos por cluster: NO MEDIDO "
          "(hace falta 1_global o 2_global_con_cluster como referencia).")
    print(f"   El WAPE de 3_por_cluster es {_pc:.5f} y el baseline {_b:.5f}: "
          f"{100*(_b-_pc)/_b:+.1f}%." if _pc is not None else "")
    print("   Eso dice cuanto da, no si partir por cluster fue una buena idea.")

pl.DataFrame([{'rama': r, **METRICAS[r]} for r in RAMAS]).write_csv(DIR_OUT / 'ramas.csv')

## 8 — ¿Dónde gana cada rama?

El WAPE global puede esconder que la rama por cluster gana mucho en unos grupos y pierde
en otros. Si eso pasa, lo razonable no es elegir una rama entera: es un **híbrido**, con
modelo propio donde conviene y global donde no. La tabla de abajo dice si vale la pena.

In [ ]:
det = (TE.select('product_id', 'customer_id', 'cluster', 'clase_tn')
         .with_columns(*[pl.Series(r, pred_test[r]) for r in RAMAS]))

filas = []
for c in sorted(det['cluster'].unique().to_list()):
    b = det.filter(pl.col('cluster') == c)
    if b.height < 50:
        continue
    f = {'cluster': c, 'filas': b.height,
         'productos': b['product_id'].n_unique(),
         'tn_real': round(float(b['clase_tn'].sum()), 1)}
    for r in RAMAS:
        f[r] = round(wape(b['clase_tn'], b[r], b['product_id']), 4)
    # La ventaja necesita una rama global de referencia: si no se corrio, la columna
    # no existe y la tabla queda igual pero sin el veredicto por cluster.
    if _nom_ref in RAMAS and '3_por_cluster' in RAMAS:
        f['ventaja_por_cluster'] = round(f[_nom_ref] - f['3_por_cluster'], 4)
    filas.append(f)

por_cl = pl.DataFrame(filas)
if 'ventaja_por_cluster' in por_cl.columns:
    por_cl = por_cl.sort('ventaja_por_cluster', descending=True)
print(por_cl)
por_cl.write_csv(DIR_OUT / 'por_cluster.csv')

if 'ventaja_por_cluster' in por_cl.columns:
    print(f"\nventaja_por_cluster > 0 -> a ESE cluster le conviene su propio modelo "
          f"(referencia: {_nom_ref}).")
    _gan = por_cl.filter(pl.col('ventaja_por_cluster') > 0)['cluster'].to_list()
    print(f"Clusters donde el modelo propio gana: {_gan}")
    if 0 < len(_gan) < por_cl.height:
        print("Un hibrido (modelo propio solo en esos, global en el resto) puede ganarle "
              "a las dos ramas por separado.\nSe implementa poniendo min_filas_cluster "
              "muy alto y una lista explicita de clusters con modelo propio.")
else:
    _gan = []
    print("\nSin rama global de referencia no hay 'ventaja_por_cluster': la tabla dice "
          "el WAPE de cada\ncluster, no si a ese cluster le conviene modelo propio. "
          "Los WAPE entre clusters NO son\ncomparables entre si -- cada uno se agrega "
          "sobre sus propios productos, que son distintos.")

# ── ¿Usa el modelo las features de cluster? ──────────────────────────────
# Si hay modelo global se mira ese. Si no, se SUMA el gain de todos los modelos por
# cluster: comparten la misma lista de features, asi que la suma es una importancia
# agregada legitima -- y es la unica disponible cuando no se corrio ninguna global.
_m_imp = mod_test.get('2_global_con_cluster') or mod_test.get('1_global')
if _m_imp is not None:
    _nombres = _m_imp.feature_name_
    _gain = _m_imp.booster_.feature_importance('gain')
    _de = f"modelo {'2_global_con_cluster' if '2_global_con_cluster' in mod_test else '1_global'}"
else:
    _mods_cl = mod_test['3_por_cluster'][0]
    if not _mods_cl:
        raise RuntimeError("no hay ningun modelo del que sacar importancia")
    _primero = next(iter(_mods_cl.values()))
    _nombres = _primero.feature_name_
    _gain = np.zeros(len(_nombres), dtype=np.float64)
    for _mm in _mods_cl.values():
        _gain += _mm.booster_.feature_importance('gain')
    _de = f"suma de los {len(_mods_cl)} modelos por cluster"
print(f"Importancia de: {_de}")

imp = (pl.DataFrame({'feature': _nombres, 'gain': _gain})
         .with_columns((100 * pl.col('gain') / pl.col('gain').sum()).round(3).alias('gain_pct'))
         .sort('gain', descending=True)
         .with_row_index('puesto'))
imp.write_csv(DIR_OUT / 'importancia.csv')

print("\nLas features que aporta el cluster, y en que puesto quedaron:")
_feat_cl = [c for c in FEATURES if 'cluster' in c]
print(imp.filter(pl.col('feature').is_in(_feat_cl)))
print(f"\n(de {imp.height} features en total)")
print("Si estan todas al fondo, el cluster no aporta nada que los otros niveles no "
      "dijeran ya.")
print("OJO con sh_cluster: tn_cluster es la suma de una fraccion grande del mercado, "
      "asi que\nsh_cluster codifica en parte el TAMANIO del par y puede aparecer alta "
      "aunque el\nclustering no sirva. La prueba es la comparacion de ramas, no la "
      "importancia.")
print("\nTOP 15 general:")
print(imp.head(15).select('puesto', 'feature', 'gain_pct'))

## 9 — Inferencia y submit

La predicción se hace a nivel par y **después** se suma por producto. Dos cosas que
tienen que salir bien acá:

- **La inferencia no se filtra nunca por cliente.** Si un par no tiene predicción, su
  producto queda subestimado en el submit, y eso es un error silencioso que no aparece
  en ninguna métrica.
- **Los 780 productos de la lista oficial tienen que estar.** Los que no tengan ninguna
  fila de inferencia van en 0, y el notebook avisa cuántos son. Si son más del 5% hay
  que revisar la densificación antes de subir.

In [ ]:
infer = df.filter(pl.col('periodo').is_in(MESES_INFER))
print(f"filas de inferencia: {infer.height:,}   "
      f"({infer.select(KEYS).n_unique():,} pares, "
      f"{infer['product_id'].n_unique()} productos)")

MESES_TODOS = sorted(periodos_sup)
print(f"\nreentrenando el ganador ({GANADOR}) con los {len(MESES_TODOS)} meses "
      f"supervisados...")
t0 = time.time()

if GANADOR == '0_baseline':
    pred_final = baseline(infer)
elif GANADOR == '1_global':
    pred_final = predecir(entrenar(bloque(MESES_TODOS), FEAT_SIN_CL,
                                    HIPER.get('global_sin_cluster')), infer, FEAT_SIN_CL)
elif GANADOR == '2_global_con_cluster':
    pred_final = predecir(entrenar(bloque(MESES_TODOS), FEAT_CON_CL,
                                    HIPER.get('global_con_cluster')), infer, FEAT_CON_CL)
else:
    _mc, _mg, _ch = entrenar_por_cluster(MESES_TODOS, FEAT_SIN_CL)
    pred_final = predecir_por_cluster(_mc, _mg, infer, FEAT_SIN_CL)

pred_final = np.maximum(pred_final, PARAM['clip_min'])
print(f"[{time.time()-t0:.0f}s]")

pred_par = infer.select(KEYS + ['periodo', 'periodo_objetivo', 'cluster']).with_columns(
    pl.Series('tn_pred', pred_final))
pred_par.write_parquet(DIR_OUT / 'predicciones_par.parquet')

# ── De par a producto: la suma ───────────────────────────────────────────
OBJ = PARAM['periodo_objetivo']
obj = pred_par.filter(pl.col('periodo_objetivo') == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_par['periodo_objetivo'].unique().to_list())}")

por_prod = obj.group_by('product_id').agg(pl.col('tn_pred').sum().alias('tn'))
print(f"\nmes objetivo {OBJ}: {obj.height:,} pares -> {por_prod.height} productos")

oficiales = pl.read_csv(DIR_RAW / 'product_id_apredecir201912.txt')
submit = oficiales.select('product_id').join(por_prod, on='product_id', how='left')
sin_pred = int(submit['tn'].null_count())
submit = submit.with_columns(pl.col('tn').fill_null(0.0)).sort('product_id')

print(f"\nSubmit: {submit.height} filas   sin prediccion (van en 0): {sin_pred}")
if sin_pred > oficiales.height * 0.05:
    print("   ATENCION: mas del 5% de la lista sin prediccion. Revisa la densificacion.")
print(f"tn  min {submit['tn'].min():.3f}  media {submit['tn'].mean():.3f}  "
      f"max {submit['tn'].max():.3f}  suma {submit['tn'].sum():,.1f}")

path_submit = DIR_OUT / f'submission_{OBJ}.csv'
submit.write_csv(path_submit)
shutil.copy(path_submit, RUTA_EXP / 'submission_ultima.csv')
print(f"Guardado: {path_submit}")

In [ ]:
def kaggle_cli(args):
    try:
        r = subprocess.run(['kaggle'] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or '') + (r.stderr or '')
    except FileNotFoundError:
        return False, 'La CLI de kaggle no esta instalada.  pip install kaggle'
    except Exception as e:
        return False, f'{type(e).__name__}: {e}'


if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube. El CSV ya esta generado.")
else:
    kd = Path.home() / '.kaggle' / 'kaggle.json'
    kd.parent.mkdir(parents=True, exist_ok=True)
    if not kd.exists():
        for cand in (BUCKET / 'kaggle.json', BUCKET / 'kaggle' / 'kaggle.json'):
            if cand.exists():
                shutil.copy(cand, kd)
                break
    if not kd.exists():
        print("Sin credenciales de Kaggle. El CSV ya esta generado.")
    else:
        kd.chmod(0o600)
        msg = f"{EXPERIMENTO} | {GANADOR} | wape_test={METRICAS[GANADOR]['test']:.5f}"
        ok, salida = kaggle_cli(['competitions', 'submit',
                                 '-c', PARAM['kaggle_competition'],
                                 '-f', str(path_submit), '-m', msg])
        print(f"{msg}\n{salida}")
        print("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco.")

## 10 — Registro y leaderboard

In [ ]:
resultado = {
    'experimento': EXPERIMENTO,
    'idea': ('el cluster de dtw_nuevo como nivel de agregacion en el FE, y un modelo '
             'por cluster; se compara contra un modelo global con y sin la etiqueta'),
    'archivo_clusters': PATH_CL.name,
    'config_clusters': _cfg_cl,
    'fuente': path_src.name,
    'solo_productos_target': PARAM['solo_productos_target'],
    'top_clientes': PARAM['top_clientes'],
    'horizonte': H, 'max_lags': L, 'lags_agregado': LA,
    'niveles_share': list(NIV),
    'min_filas_cluster': PARAM['min_filas_cluster'],
    'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
    'meses_inferencia': MESES_INFER, 'periodo_objetivo': OBJ,
    'n_clusters': int(df['cluster'].n_unique()),
    'clusters_con_modelo_propio': (sorted(int(c) for c in mod_test['3_por_cluster'][0])
                                   if '3_por_cluster' in mod_test else []),
    'clusters_al_global': ([int(c) for c in mod_test['3_por_cluster'][2]]
                           if '3_por_cluster' in mod_test else []),
    'n_features': len(FEATURES), 'features': FEATURES, 'cat_features': CAT_FEATURES,
    'metricas_por_rama': METRICAS, 'rama_ganadora': GANADOR,
    'ramas_corridas': list(RAMAS),
    'aporte_cluster_feature_pct': (round(APORTE_FEATURE, 3)
                                   if APORTE_FEATURE is not None else None),
    'aporte_partir_por_cluster_pct': (round(APORTE_PARTIR, 3)
                                      if APORTE_PARTIR is not None else None),
    'referencia_de_la_comparacion': _nom_ref if APORTE_PARTIR is not None else None,
    'por_cluster': por_cl.to_dicts(),
    'n_filas_supervisadas': int(sup.height),
    'n_sin_prediccion': sin_pred,
    'tn_total': float(submit['tn'].sum()),
    'optuna_trials': PARAM['optuna_trials'],
    'estudios_optuna': ESTUDIOS,
    'hiperparametros_por_modelo': HIPER,
    'lgbm': PARAM['lgbm'], 'semilla': PARAM['semilla'],
}
with open(DIR_OUT / 'resultado.json', 'w', encoding='utf-8') as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

fila = {'experimento': EXPERIMENTO, 'clusters': PATH_CL.stem,
        'n_clusters': resultado['n_clusters'], 'ganador': GANADOR,
        **{f'test_{r}': round(METRICAS[r]['test'], 5) for r in RAMAS},
        **{f'val_{r}': round(METRICAS[r]['val'], 5) for r in RAMAS},
        'aporte_feature_pct': resultado['aporte_cluster_feature_pct'],
        'aporte_partir_pct': resultado['aporte_partir_por_cluster_pct'],
        'ramas': '+'.join(RAMAS),
        'sin_prediccion': sin_pred, 'tn_total': round(float(submit['tn'].sum()), 1)}
path_lb = RUTA_EXP / 'leaderboard.csv'
nueva = pl.DataFrame([fila])
if path_lb.exists():
    viejo = pl.read_csv(path_lb).filter(pl.col('experimento') != EXPERIMENTO)
    nueva = pl.concat([viejo, nueva], how='diagonal_relaxed')
_orden = ('test_3_por_cluster' if 'test_3_por_cluster' in nueva.columns
          else 'test_0_baseline')
nueva.sort(_orden).write_csv(path_lb)

print(f"Archivos en {DIR_OUT.relative_to(BUCKET)}:")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nleaderboard ({nueva.height} experimentos):")
_cols_lb = [c for c in ('n_clusters', 'ganador', 'ramas', 'test_0_baseline',
                        'test_1_global', 'test_2_global_con_cluster',
                        'test_3_por_cluster', 'aporte_feature_pct',
                        'aporte_partir_pct') if c in nueva.columns]
print(nueva.select(_cols_lb))

## 11 — Cómo leer el resultado

**Primero `aporte_feature_pct` (rama 2 vs 1).** ¿Sirve saber a qué cluster pertenece un
par? Si es positivo, el clustering produjo información utilizable y ya justificó el
trabajo de `dtw_nuevo` — sin necesidad de partir nada. Mirá también en qué puesto de la
importancia quedaron `sh_cluster`, `tn_cluster_lag*` y `cluster`: si están al fondo, el
cluster no dice nada que `sh_producto` o `sh_cat3` no dijeran ya.

**Después `aporte_partir_pct` (rama 3 vs 2).** ¿Conviene un modelo por grupo? Ojo con la
tentación de leer sólo esto: es la comparación difícil de ganar, porque partir los datos
le quita filas a cada modelo mientras el árbol de la rama 2 ya podía separar por cluster
cuando le convenía. **Un resultado negativo acá es el esperado**, y no invalida el
clustering — invalida la partición como estrategia de entrenamiento.

**Si `por_cluster.csv` muestra que la ventaja se concentra** en dos o tres clusters, ahí
está el híbrido: modelo propio sólo donde gana. Suele ser mejor que las dos ramas
enteras, porque los clusters donde el especializado pierde son justamente los que tienen
pocas filas o forma poco definida.

**Y un chequeo de honestidad antes de subir.** El `sin_prediccion` del submit tiene que
ser chico. Si es grande, el problema no está en el modelo: está en que la densificación
dejó productos sin fila en el último mes, y ninguna métrica de val/test lo detecta
porque esos productos tampoco estaban ahí.